# Exploring Neural Collapse in current model

In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import os
import sys
sys.path.append('../../../')
from polygene.model.model import load_trained_model
mod, tok = load_trained_model("../../../runs/gesam_polygene_run_4/")


EMBEDDING_DIR = "/media/lleger/LaCie/mit/disease_vector/"
df_list = []
for file in os.listdir(EMBEDDING_DIR):
    if not file.startswith('embeddings'): continue
    embeddings = pd.read_pickle(EMBEDDING_DIR + file)
    df_list.append(pd.DataFrame({'embedding': embeddings[0].tolist()} | {tok.phenotypic_types[idx]:embeddings[2][:, idx] for idx in range(len(tok.phenotypic_types))}) )
    

df = pd.concat(df_list)

def neural_collapse(df, labels_column="disease"):

    K = (df[labels_column].value_counts() > 100).sum()
    diseases = df[labels_column].value_counts()[df[labels_column].value_counts() > 100].index.tolist()
    df = df[df[labels_column].isin(diseases)]
    
    print(K)

    stats_df = df.groupby(labels_column).apply(lambda g: pd.Series({'mean': np.array(g['embedding'].tolist()).mean(axis=0), 'covar': np.cov(np.array(g['embedding'].tolist()).T, bias=True)})).reset_index()
    covariance_K = np.array(stats_df['covar'].tolist()).mean(axis = 0)
    covariance_G = np.cov(np.array(stats_df['mean'].tolist()).T, bias=True)
    NC1 = (1 / K) * np.trace(covariance_K @ np.linalg.pinv(covariance_G))
    means = np.array(stats_df["mean"].tolist()) - np.array(stats_df["mean"].tolist()).mean(axis=0)
    M = means / np.linalg.norm(means, axis=1, keepdims=True)
    MMT = M @ M.T
    NC2 = np.linalg.norm(MMT / np.linalg.norm(MMT, "fro") - (1 / np.sqrt(K - 1)) * (np.eye(K) - (1 / K) * np.ones((K, K))), "fro")
    return NC1, NC2
neural_collapse(df)
# 0 in log scale is quite worrying and we should modify our loss function to address it. 

50


(1.009419700218355, 0.7950036057993237)